# 第七章 文本扩展

<div class="toc">
    <ul class="toc-item">
        <li><span><a href="#一引言" data-toc-modified-id="一、引言">一、引言</a></span></li>
        <li>
            <span><a href="#二定制客户邮件" data-toc-modified-id="二、定制客户邮件">二、定制客户邮件</a></span>
        </li>
        <li><span><a href="#三引入温度系数" data-toc-modified-id="三、引入温度系数">三、引入温度系数</a></span>
        </li>
    </ul>
</div>

## 一、引言

扩展是将短文本（例如一组说明或主题列表）输入到大型语言模型中，让模型生成更长的文本（例如基于某个主题的电子邮件或论文）。这种应用是一把双刃剑，好处例如将大型语言模型用作头脑风暴的伙伴；但也存在问题，例如某人可能会使用它来生成大量垃圾邮件。因此，当你使用大型语言模型的这些功能时，请仅以**负责任** (responsible) 和**有益于人们** (helps people) 的方式使用它们。

在本章中，你将学会如何基于 OpenAI API 生成*针对每位客户评价优化*的客服电子邮件。我们还将利用模型的另一个输入参数称为温度，这种参数允许您在模型响应中变化探索的程度和多样性。

同以上几章，你需要类似的代码来配置一个可以使用 OpenAI API 的环境

In [15]:
import os
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown

loaded = load_dotenv(find_dotenv(), override=True)
# 从环境变量中获取 OpenAI API Key 或者直接赋值
API_KEY = os.getenv("API_KEY")

# 如果您使用的是官方 API，就直接用 https://api.siliconflow.cn/v1 就行。
BASE_URL = "https://api.siliconflow.cn/v1"



In [2]:
# 实例化 OpenAI 对象
# 传入参数：OpenAI API Key（必需）、Base URL 和最大重试次数
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=3)

In [3]:
# 参数 n，整数或 Null，可选项，默认为 1。为每条输入信息生成多少个聊天完成选项。
# 参数 temperature，实数值或 Null，可选项，默认为 1。使用的采样温度，介于 0 和 2 之间。0.8 等较高值会使输出更加随机，而 0.2 等较低值会使输出更加集中和确定。

def get_completions(llm_prompt, model_endpoint, temperature=0):
    extra_body = {}
    if "Qwen3" in model_endpoint:
        extra_body={
            "enable_thinking": False
        }
        
    response = client.chat.completions.create(model=model_endpoint,
                                              messages=[
                                                        {"role": "user",
                                                         "content": llm_prompt
                                                        }
                                                       ],
                                              n=1, temperature=temperature, seed=42,
                                              presence_penalty=0, frequency_penalty=0,
                                              max_tokens=512, extra_body = extra_body
                                             )

    return response.choices[0].message.content.strip()

## 二、定制客户邮件

我们将根据客户评价和情感，针对性写自动回复邮件。因此，我们将给定客户评价和情感，使用 LLM 针对性生成响应，即根据客户评价和评论情感生成定制电子邮件。

我们首先给出一个示例，包括一个评论及对应的情感。

In [8]:
# given the sentiment from the lesson on "inferring",
# and the original customer message, customize the email
sentiment = "negative"

# review for a blender
review_en = f"""
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

In [11]:
# 我们可以在推理那章学习到如何对一个评论判断其情感倾向
sentiment = "negative"

# 一个产品的评价
review_zh = f"""
他们在11月份的季节性销售期间以约49美元的价格出售17件套装，折扣约为一半。\
但由于某些原因（可能是价格欺诈），到了12月第二周，同样的套装价格全都涨到了70美元到89美元不等。\
11件套装的价格也上涨了大约10美元左右。\
虽然外观看起来还可以，但基座上锁定刀片的部分看起来不如几年前的早期版本那么好。\
不过我打算非常温柔地使用它，例如，\
我会先在搅拌机中将像豆子、冰、米饭等硬物研磨，然后再制成所需的份量，\
切换到打蛋器制作更细的面粉，或者在制作冰沙时先使用交叉切割刀片，然后使用平面刀片制作更细/不粘的效果。\
制作冰沙时，特别提示：\
将水果和蔬菜切碎并冷冻（如果使用菠菜，则轻轻煮软菠菜，然后冷冻直到使用；\
如果制作果酱，则使用小到中号的食品处理器），这样可以避免在制作冰沙时添加太多冰块。\
大约一年后，电机发出奇怪的噪音，我打电话给客服，但保修已经过期了，所以我不得不再买一个。\
总的来说，这些产品的总体质量已经下降，因此它们依靠品牌认可和消费者忠诚度来维持销售。\
货物在两天内到达。
"""

我们已经使用推断课程中所学方法提取了情感，这是一个关于搅拌机的客户评价，现在我们将根据情感定制回复。

以下述 Prompt 为例：假设你是一个客户服务 AI 助手，你的任务是为客户发送电子邮件回复，根据通过三个反引号分隔的客户电子邮件，生成一封回复以感谢客户的评价。

In [9]:
llm = "Qwen/Qwen3-8B"

prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review_en}```
Review sentiment: {sentiment}
"""
response = get_completions(prompt, llm)
print(response)

Subject: Thank You for Your Feedback

Dear [Customer's Name],

Thank you for taking the time to share your review and experience with our product. We sincerely appreciate your honesty and detailed input.

We understand that you noticed a price increase for the 17-piece system around the second week of December, which was unexpected after the seasonal sale in November. Additionally, you mentioned that the base of the blender, where the blade locks into place, does not appear as durable as in previous editions. We regret that the motor began making a funny noise after about a year, and we are sorry to hear that the warranty had expired by then, requiring you to purchase a replacement.

We take your concerns about the overall quality of these types of products seriously and are aware that brand recognition and consumer loyalty play a role in maintaining sales. Thank you for bringing this to our attention.

If you have any further questions or need assistance, please do not hesitate to rea

In [12]:
llm = "Qwen/Qwen3-8B"

prompt = f"""
你是一位客户服务的AI助手。
你的任务是给一位重要客户发送邮件回复。
根据客户通过“```”分隔的评价，生成回复以感谢客户的评价。提醒模型使用评价中的具体细节
用简明而专业的语气写信。
作为“AI客户代理”签署电子邮件。
客户评论：
```{review_zh}```
评论情感：{sentiment}
"""
response = get_completions(prompt, llm)
print(response)

主题：感谢您的反馈与建议

尊敬的客户，

感谢您花时间分享您的使用体验。我们非常重视您的意见，并对您在11月份购买17件套装时所获得的优惠表示赞赏，当时的价格约为49美元，确实是一个极具吸引力的促销活动。

然而，我们也注意到您提到在12月第二周，同样的套装价格有所上涨，达到了70美元到89美元不等，而11件套装的价格也上涨了约10美元。我们理解您对价格变动的担忧，尤其是您提到可能存在价格欺诈的情况，对此我们深表歉意，并会将您的反馈转达给相关部门，以期在未来的促销活动中更加透明和合理。

此外，您提到基座上锁定刀片的部分不如早期版本稳固，我们对此表示理解。虽然我们努力在产品设计中保持高品质，但我们也意识到随着产品迭代，部分用户可能会对新版本的使用体验有所期待。感谢您提出这一细节，我们将认真考虑如何在后续产品中进一步优化这一设计。

您分享的使用技巧非常实用，尤其是关于如何通过预处理食材来延长刀片使用寿命和提升冰沙制作效果的方法。我们非常欣赏您对产品的深入理解和细致使用方式。

遗憾的是，您提到在使用约一年后电机发出奇怪的噪音，且由于保修已过期，您不得不重新购买。我们对此表示诚挚的歉意，并希望未来能为您提供更持久可靠的产品。您的反馈对我们改进产品和服务至关重要。

再次感谢您对我们产品的支持与信任，也感谢您提出宝贵的建议。我们期待有机会为您提供更好的服务，并希望您能继续关注我们的产品。

此致  
敬礼  

AI客户代理


## 三、引入温度系数

接下来，我们将使用语言模型的一个称为“温度” (Temperature) 的参数，它将允许我们改变模型响应的多样性。您可以将温度视为模型探索或随机性的程度。

例如，在一个特定的短语中，“我的最爱食品”最有可能的下一个词是“比萨”，其次最有可能的是“寿司”和“塔可”。因此，在温度为零时，模型将总是选择最有可能的下一个词，而在较高的温度下，它还将选择其中一个不太可能的词，在更高的温度下，它甚至可能选择塔可，而这种可能性仅为五分之一。您可以想象，随着模型继续生成更多单词的最终响应，“我的最爱食品是比萨”将会与第一个响应“我的最爱食品是塔可”产生差异。随着模型的继续，这两个响应也将变得越来越不同。

一般来说，在构建需要可预测响应的应用程序时，我建议**设置温度为零**。在所有课程中，我们一直设置温度为零，如果您正在尝试构建一个可靠和可预测的系统，我认为您应该选择这个温度。如果您尝试以更具创意的方式使用模型，可能需要更广泛地输出不同的结果，那么您可能需要使用更高的温度。

同一段来信，我们提醒模型使用用户来信中的详细信息，并设置温度：

In [13]:
llm = "Qwen/Qwen3-8B"

prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review_en}```
Review sentiment: {sentiment}
"""
response = get_completions(prompt, llm, temperature=0.7)
print(response)

Subject: Thank You for Your Feedback

Dear Valued Customer,

Thank you for taking the time to share your review. We sincerely appreciate your input, even though we understand that your experience has been somewhat disappointing.

We noticed your concern regarding the price increase of the 17-piece system from approximately $49 in November to $70–$89 by late December, as well as the $10 increase for the 11-piece system. We regret that the pricing changes may have affected your satisfaction, and we are sorry to hear that the warranty had expired by the time the motor began making unusual noises, which required you to purchase a replacement.

Your detailed insight about the base design and the blade locking mechanism is also valuable. We acknowledge that the quality of the product may have changed from previous editions, and we appreciate your mention of how you adapt your usage to maintain performance.

Please know that we value your loyalty and are committed to improving our offerings. 

In [14]:
llm = "Qwen/Qwen3-8B"

prompt = f"""
你是一名客户服务的AI助手。
你的任务是给一位重要的客户发送邮件回复。
根据通过“```”分隔的客户电子邮件生成回复，以感谢客户的评价。
如果情感是积极的或中性的，感谢他们的评价。
如果情感是消极的，道歉并建议他们联系客户服务。
请确保使用评论中的具体细节。
以简明和专业的语气写信。
以“AI客户代理”的名义签署电子邮件。
客户评价：```{review_zh}```
评论情感：{sentiment}
"""
response = get_completions(prompt, llm, temperature=0.7)
print(response)

主题：感谢您的反馈 - 我们重视您的意见

尊敬的客户，

感谢您花时间分享您的使用体验和反馈。我们非常重视您的意见，并对您提到的产品在价格和质量方面的变化感到遗憾。

您提到在11月份的季节性销售期间，17件套装以约49美元的价格出售，折扣约为一半，但到了12月第二周，同样的套装价格却上涨到了70美元到89美元不等，11件套装的价格也上涨了约10美元。这可能是由于库存调整或其他原因导致的，我们理解您对此感到不满。

此外，您指出基座上锁定刀片的部分不如早期版本牢固，这确实可能影响使用体验。不过您计划通过合理的方式使用产品，如先处理硬物再制作细腻的面粉，以及在制作冰沙时采用分阶段处理的方法，我们非常欣赏您对产品的细心使用和维护。

关于电机在使用一年后发出奇怪的噪音，虽然您提到保修已过期，但我们也理解您的困扰。我们建议您在未来的购买中，留意产品的保修期限，并在遇到任何问题时及时联系我们的客户服务团队，以便获得更好的支持。

总体而言，您对产品质量下降的观察是准确的，我们深知品牌认可和消费者忠诚度的重要性，并将持续努力提升产品质量与服务体验。

再次感谢您的反馈，希望您未来能有更满意的购物体验。

此致  
敬礼  

AI客户代理


在温度为零时，每次执行相同的 Prompt ，您获得的回复理应相同。而使用温度为 0.7 时，则每次都会获得不同的输出。

所以，您可以看到它与我们之前收到的电子邮件不同。再次执行将再次获得不同的电子邮件。

因此，我建议您自己尝试温度，以查看输出如何变化。总之，在更高的温度下，模型的输出更加随机。您几乎可以将其视为在更高的温度下，助手**更易分心**，但也许**更有创造力**。